In [ ]:
!pip install -q faster-whisper
!nvidia-smi | head -n 12

In [ ]:
import os, pandas as pd
base = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files:
        base = root
print(base)
train = pd.read_csv(f'{base}/train.csv')
test = pd.read_csv(f'{base}/test.csv')
print(train.head())
print(train.columns.tolist(), len(train), len(test))

In [ ]:
wav = {'train': {}, 'test': {}}
for root, _, files in os.walk(base):
    split = 'train' if '/train' in root else 'test' if '/test' in root else None
    if split is None:
        continue
    for f in files:
        if f.endswith('.wav'):
            wav[split][f] = os.path.join(root, f)
print(len(wav['train']), len(wav['test']))   # 769 216 undali

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import librosa
from tqdm import tqdm
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cuda", compute_type="float16")

out_path = "/kaggle/working/transcripts.csv"
if os.path.exists(out_path):
    done = pd.read_csv(out_path)
else:
    done = pd.DataFrame(columns=["split", "filename", "text"])
results = done.to_dict("records")
finished = set(zip(done["split"], done["filename"]))

items = [(s, n, p) for s in ["train", "test"] for n, p in wav[s].items()]

for i, (s, n, p) in enumerate(tqdm(items)):
    if (s, n) in finished:
        continue
    audio, _ = librosa.load(p, sr=16000)   # 16 kHz mono float32, Whisper ki ide kavali
    segs, _ = model.transcribe(audio, language="en", beam_size=5,
                               initial_prompt="Umm, so, uh, you know, I mean, like.")
    text = " ".join(x.text.strip() for x in segs)
    results.append({"split": s, "filename": n, "text": text})
    if i % 50 == 0:
        pd.DataFrame(results).to_csv(out_path, index=False)

pd.DataFrame(results).to_csv(out_path, index=False)
print("done", len(results))

In [ ]:
t = pd.read_csv('/kaggle/working/transcripts.csv')
print(t.shape)
print(t['text'].str.split().str.len().describe())
print(t.iloc[0]['text'][:300])

In [ ]:
t = pd.read_csv('/kaggle/working/transcripts.csv').fillna('')
t['n_words'] = t['text'].str.split().str.len()
print(t[t.n_words < 30][['split', 'filename', 'n_words', 'text']])

sub = pd.read_csv(f'{base}/sample_submission.csv')
print(sub.head(), sub.columns.tolist())

tr = train.merge(t[t.split == 'train'][['filename', 'text']], on='filename')
te = test.merge(t[t.split == 'test'][['filename', 'text']], on='filename')
print(tr.shape, te.shape)

In [ ]:
import re, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# 1) Sentence embeddings (captures sentence structure and fluency)
emb_model = SentenceTransformer('all-mpnet-base-v2', device='cuda')
E_tr = emb_model.encode(tr.text.tolist(), batch_size=32, show_progress_bar=True)
E_te = emb_model.encode(te.text.tolist(), batch_size=32, show_progress_bar=True)

# 2) Simple handcrafted features
def feats(s):
    w = s.split()
    sents = [x for x in re.split(r'[.!?]+', s) if x.strip()]
    fill = len(re.findall(r'\b(um+|uh+|like|you know)\b', s.lower()))
    return [len(w), len(sents), len(w) / max(len(sents), 1),
            len(set(w)) / max(len(w), 1), fill,
            np.mean([len(x) for x in w]) if w else 0]

F_tr = np.array([feats(s) for s in tr.text])
F_te = np.array([feats(s) for s in te.text])
sc = StandardScaler().fit(F_tr)

X_tr = np.hstack([E_tr, sc.transform(F_tr)])
X_te = np.hstack([E_te, sc.transform(F_te)])
y = tr.label.values

# 3) 5-fold CV for different alphas
kf = KFold(5, shuffle=True, random_state=42)
print("Mean-predictor RMSE (reference):", round(y.std(), 3))
best = None
for a in [1, 3, 10, 30, 100]:
    oof = cross_val_predict(Ridge(alpha=a), X_tr, y, cv=kf)
    rmse = mean_squared_error(y, oof) ** 0.5
    r = pearsonr(y, oof)[0]
    print(f"alpha={a}  CV RMSE={rmse:.3f}  Pearson={r:.3f}")
    if best is None or rmse < best[1]:
        best = (a, rmse)

# 4) Final fit, training RMSE (compulsory in notebook), predictions
model = Ridge(alpha=best[0]).fit(X_tr, y)
train_rmse = mean_squared_error(y, model.predict(X_tr)) ** 0.5
print("Best alpha:", best[0], "| Training RMSE:", round(train_rmse, 3))

pred = np.clip(model.predict(X_te), 0, 5)
out = pd.DataFrame({'filename': te.filename, 'label': pred})
out.to_csv('/kaggle/working/submission.csv', index=False)
print(out.head())

In [ ]:
from sklearn.svm import SVR

def cv(model, X):
    oof = cross_val_predict(model, X, y, cv=kf)
    return round(mean_squared_error(y, oof) ** 0.5, 3), round(pearsonr(y, oof)[0], 3)

for a in [0.1, 0.3, 1]:
    print("Ridge", a, cv(Ridge(alpha=a), X_tr))
for C in [1, 3, 10]:
    print("SVR", C, cv(SVR(C=C), X_tr))
print(sorted(set(y)))   # label range check, clip kosam


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained('gpt2')
lm = GPT2LMHeadModel.from_pretrained('gpt2').cuda().eval()

def log_ppl(s):
    ids = tok(s, return_tensors='pt', truncation=True, max_length=512).input_ids.cuda()
    if ids.shape[1] < 2:
        return np.nan
    with torch.no_grad():
        return float(lm(ids, labels=ids).loss)

P_tr = np.array([log_ppl(s) for s in tr.text])
P_te = np.array([log_ppl(s) for s in te.text])
med = np.nanmedian(P_tr)
P_tr, P_te = np.nan_to_num(P_tr, nan=med), np.nan_to_num(P_te, nan=med)

sc2 = StandardScaler().fit(P_tr.reshape(-1, 1))
X2_tr = np.hstack([X_tr, sc2.transform(P_tr.reshape(-1, 1))])
X2_te = np.hstack([X_te, sc2.transform(P_te.reshape(-1, 1))])
for a in [0.3, 1]:
    print("Ridge + perplexity", a, cv(Ridge(alpha=a), X2_tr))

In [ ]:
import torch, librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base-960h')
w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').cuda().eval().half()

def audio_emb(path):
    a, _ = librosa.load(path, sr=16000)
    x = fe(a, sampling_rate=16000, return_tensors='pt').input_values.cuda().half()
    with torch.no_grad():
        hs = w2v(x, output_hidden_states=True).hidden_states
    # middle layer + last layer, mean pooled over time
    return torch.cat([hs[6].mean(1), hs[-1].mean(1)], dim=1).float().cpu().numpy()[0]

A_tr = np.array([audio_emb(wav['train'][f]) for f in tqdm(tr.filename)])
A_te = np.array([audio_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/A_tr.npy', A_tr); np.save('/kaggle/working/A_te.npy', A_te)

sa = StandardScaler().fit(A_tr)
X3_tr = np.hstack([X_tr, sa.transform(A_tr)])
X3_te = np.hstack([X_te, sa.transform(A_te)])
for a in [10, 30, 100, 300]:
    print("Ridge + audio", a, cv(Ridge(alpha=a), X3_tr))

In [ ]:
best_a, best_r = None, 9
for a in [300, 1000, 3000, 10000]:
    r, p = cv(Ridge(alpha=a), X3_tr)
    print(f"Ridge + audio alpha={a}  CV RMSE={r}  Pearson={p}")
    if r < best_r:
        best_a, best_r = a, r

model = Ridge(alpha=best_a).fit(X3_tr, y)
train_rmse = mean_squared_error(y, model.predict(X3_tr)) ** 0.5
print("Best alpha:", best_a, "| Training RMSE:", round(train_rmse, 3))

pred = np.clip(model.predict(X3_te), 0, 5)
out = pd.DataFrame({'filename': te.filename, 'label': pred})
out.to_csv('/kaggle/working/submission_v2.csv', index=False)
print(out.shape, out.label.describe())

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel

fe2 = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base-plus').cuda().eval().half()

def wavlm_emb(path):
    a, _ = librosa.load(path, sr=16000)
    x = fe2(a, sampling_rate=16000, return_tensors='pt').input_values.cuda().half()
    with torch.no_grad():
        hs = wavlm(x, output_hidden_states=True).hidden_states
    feats_ = []
    for l in [4, 8, 12]:                       # low, mid, top layers
        h = hs[l][0].float()
        feats_ += [h.mean(0), h.std(0)]        # mean + std over time
    return torch.cat(feats_).cpu().numpy()

W_tr = np.array([wavlm_emb(wav['train'][f]) for f in tqdm(tr.filename)])
W_te = np.array([wavlm_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/W_tr.npy', W_tr); np.save('/kaggle/working/W_te.npy', W_te)

sw = StandardScaler().fit(W_tr)
X4_tr = np.hstack([X3_tr, sw.transform(W_tr)])
X4_te = np.hstack([X3_te, sw.transform(W_te)])
for a in [1000, 3000, 10000, 30000]:
    print("Ridge + wav2vec2 + WavLM", a, cv(Ridge(alpha=a), X4_tr))

In [ ]:
best_a, best_r = None, 9
for a in [200, 300, 500, 1000]:
    r, p = cv(Ridge(alpha=a), X4_tr)
    print(f"Ridge + text + wav2vec2 + WavLM alpha={a}  CV RMSE={r}  Pearson={p}")
    if r < best_r:
        best_a, best_r = a, r

model = Ridge(alpha=best_a).fit(X4_tr, y)
train_rmse = mean_squared_error(y, model.predict(X4_tr)) ** 0.5
print("Best alpha:", best_a, "| Training RMSE:", round(train_rmse, 3))

pred = np.clip(model.predict(X4_te), 0, 5)
out = pd.DataFrame({'filename': te.filename, 'label': pred})
out.to_csv('/kaggle/working/submission_v3.csv', index=False)
print(out.shape, out.label.describe())

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperModel

wfe = WhisperFeatureExtractor.from_pretrained('openai/whisper-small')
wenc = WhisperModel.from_pretrained('openai/whisper-small').encoder.cuda().eval().half()

def whisper_emb(path):
    a, _ = librosa.load(path, sr=16000)
    chunks = [a[i:i + 480000] for i in range(0, len(a), 480000)]   # 30 sec chunks
    chunks = [c for c in chunks if len(c) > 16000] or [a]
    embs = []
    for c in chunks:
        n = max(int(len(c) / 16000 * 50), 2)        # 50 frames per sec
        x = wfe(c, sampling_rate=16000, return_tensors='pt').input_features.cuda().half()
        with torch.no_grad():
            hs = wenc(x, output_hidden_states=True).hidden_states
        f = []
        for l in [6, 9, 12]:
            h = hs[l][0, :n].float()
            f += [h.mean(0), h.std(0)]
        embs.append(torch.cat(f).cpu().numpy())
    return np.mean(embs, axis=0)

H_tr = np.array([whisper_emb(wav['train'][f]) for f in tqdm(tr.filename)])
H_te = np.array([whisper_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/H_tr.npy', H_tr); np.save('/kaggle/working/H_te.npy', H_te)

sh = StandardScaler().fit(H_tr)
X5_tr = np.hstack([X4_tr, sh.transform(H_tr)])
X5_te = np.hstack([X4_te, sh.transform(H_te)])
for a in [1000, 3000, 10000]:
    print("Ridge + all + Whisper enc", a, cv(Ridge(alpha=a), X5_tr))

In [ ]:
# 1) v4: all features, one Ridge (alpha 3000)
model = Ridge(alpha=3000).fit(X5_tr, y)
print("Training RMSE (v4):", round(mean_squared_error(y, model.predict(X5_tr)) ** 0.5, 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(model.predict(X5_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v4.csv', index=False)

# 2) Stacking: one Ridge per feature block, then combine their predictions
blocks_tr = {'text': X_tr, 'w2v': sa.transform(A_tr), 'wavlm': sw.transform(W_tr), 'whisper': sh.transform(H_tr)}
blocks_te = {'text': X_te, 'w2v': sa.transform(A_te), 'wavlm': sw.transform(W_te), 'whisper': sh.transform(H_te)}

oof, tep = {}, {}
for k in blocks_tr:
    scores = [(cv(Ridge(alpha=a), blocks_tr[k])[0], a) for a in [10, 100, 1000, 3000, 10000]]
    rmse_k, a_k = min(scores)
    oof[k] = cross_val_predict(Ridge(alpha=a_k), blocks_tr[k], y, cv=kf)
    tep[k] = Ridge(alpha=a_k).fit(blocks_tr[k], y).predict(blocks_te[k])
    print(f"{k}: best alpha={a_k}  CV RMSE={rmse_k:.3f}")

S_tr = np.column_stack(list(oof.values()))
S_te = np.column_stack(list(tep.values()))
print("Stack CV (RMSE, Pearson):", cv(Ridge(alpha=1), S_tr))

meta = Ridge(alpha=1).fit(S_tr, y)
print("Block weights:", dict(zip(oof.keys(), meta.coef_.round(3))))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta.predict(S_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v4_stack.csv', index=False)

In [19]:
fe3 = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-large')
wl = AutoModel.from_pretrained('microsoft/wavlm-large').cuda().eval().half()

def wavlm_large_emb(path):
    a, _ = librosa.load(path, sr=16000)
    x = fe3(a, sampling_rate=16000, return_tensors='pt').input_values.cuda().half()
    with torch.no_grad():
        hs = wl(x, output_hidden_states=True).hidden_states
    f = []
    for l in [8, 16, 24]:
        h = hs[l][0].float()
        f += [h.mean(0), h.std(0)]
    return torch.cat(f).cpu().numpy()

L_tr = np.array([wavlm_large_emb(wav['train'][f]) for f in tqdm(tr.filename)])
L_te = np.array([wavlm_large_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/L_tr.npy', L_tr); np.save('/kaggle/working/L_te.npy', L_te)

sl = StandardScaler().fit(L_tr)
blocks_tr['wavlm_large'] = sl.transform(L_tr)
blocks_te['wavlm_large'] = sl.transform(L_te)

oof, tep = {}, {}
for k in blocks_tr:
    scores = [(cv(Ridge(alpha=a), blocks_tr[k])[0], a) for a in [10, 100, 1000, 3000, 10000]]
    rmse_k, a_k = min(scores)
    oof[k] = cross_val_predict(Ridge(alpha=a_k), blocks_tr[k], y, cv=kf)
    tep[k] = Ridge(alpha=a_k).fit(blocks_tr[k], y).predict(blocks_te[k])
    print(f"{k}: best alpha={a_k}  CV RMSE={rmse_k:.3f}")

S_tr = np.column_stack(list(oof.values()))
S_te = np.column_stack(list(tep.values()))
print("Stack CV (RMSE, Pearson):", cv(Ridge(alpha=1), S_tr))
meta = Ridge(alpha=1).fit(S_tr, y)
print("Block weights:", dict(zip(oof.keys(), meta.coef_.round(3))))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta.predict(S_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v5_stack.csv', index=False)

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            



  0%|          | 0/769 [00:00<?, ?it/s]

  0%|          | 1/769 [00:11<2:28:11, 11.58s/it]

  0%|          | 2/769 [00:12<1:09:15,  5.42s/it]

  0%|          | 3/769 [00:13<43:58,  3.44s/it]  

  1%|          | 4/769 [00:14<29:34,  2.32s/it]

  1%|          | 5/769 [00:14<21:28,  1.69s/it]

  1%|          | 6/769 [00:16<18:56,  1.49s/it]

  1%|          | 7/769 [00:16<16:07,  1.27s/it]

  1%|          | 8/769 [00:17<15:24,  1.22s/it]

  1%|          | 9/769 [00:19<14:59,  1.18s/it]

  1%|▏         | 10/769 [00:20<14:40,  1.16s/it]

  1%|▏         | 11/769 [00:21<14:27,  1.14s/it]

  2%|▏         | 12/769 [00:22<14:19,  1.13s/it]

  2%|▏         | 13/769 [00:23<14:14,  1.13s/it]

  2%|▏         | 14/769 [00:24<14:10,  1.13s/it]

  2%|▏         | 15/769 [00:25<14:07,  1.12s/it]

  2%|▏         | 16/769 [00:26<12:06,  1.04it/s]

  2%|▏         | 17/769 [00:26<09:16,  1.35it/s]

  2%|▏         | 18/769 [00:27<10:41,  1.17it/s]

  2%|▏         | 19/769 [00:28<11:43,  1.07it/s]

  3%|▎    

text: best alpha=10  CV RMSE=0.994
w2v: best alpha=1000  CV RMSE=0.642
wavlm: best alpha=1000  CV RMSE=0.564
whisper: best alpha=3000  CV RMSE=0.567
wavlm_large: best alpha=1000  CV RMSE=0.535
Stack CV (RMSE, Pearson): (0.524, np.float64(0.906))
Block weights: {'text': np.float64(0.167), 'w2v': np.float64(0.019), 'wavlm': np.float64(0.084), 'whisper': np.float64(0.238), 'wavlm_large': np.float64(0.632)}


In [20]:
wfe2 = WhisperFeatureExtractor.from_pretrained('openai/whisper-medium')
wenc2 = WhisperModel.from_pretrained('openai/whisper-medium').encoder.cuda().eval().half()

def whisper_med_emb(path):
    a, _ = librosa.load(path, sr=16000)
    chunks = [a[i:i + 480000] for i in range(0, len(a), 480000)]
    chunks = [c for c in chunks if len(c) > 16000] or [a]
    embs = []
    for c in chunks:
        n = max(int(len(c) / 16000 * 50), 2)
        x = wfe2(c, sampling_rate=16000, return_tensors='pt').input_features.cuda().half()
        with torch.no_grad():
            hs = wenc2(x, output_hidden_states=True).hidden_states
        f = []
        for l in [12, 18, 24]:
            h = hs[l][0, :n].float()
            f += [h.mean(0), h.std(0)]
        embs.append(torch.cat(f).cpu().numpy())
    return np.mean(embs, axis=0)

M_tr = np.array([whisper_med_emb(wav['train'][f]) for f in tqdm(tr.filename)])
M_te = np.array([whisper_med_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/M_tr.npy', M_tr); np.save('/kaggle/working/M_te.npy', M_te)

sm = StandardScaler().fit(M_tr)
blocks_tr['whisper_med'] = sm.transform(M_tr)
blocks_te['whisper_med'] = sm.transform(M_te)

oof, tep = {}, {}
for k in blocks_tr:
    scores = [(cv(Ridge(alpha=a), blocks_tr[k])[0], a) for a in [10, 100, 1000, 3000, 10000]]
    rmse_k, a_k = min(scores)
    oof[k] = cross_val_predict(Ridge(alpha=a_k), blocks_tr[k], y, cv=kf)
    tep[k] = Ridge(alpha=a_k).fit(blocks_tr[k], y).predict(blocks_te[k])
    print(f"{k}: best alpha={a_k}  CV RMSE={rmse_k:.3f}")

S_tr = np.column_stack(list(oof.values()))
S_te = np.column_stack(list(tep.values()))
print("Stack CV (RMSE, Pearson):", cv(Ridge(alpha=1), S_tr))
meta = Ridge(alpha=1).fit(S_tr, y)
print("Block weights:", dict(zip(oof.keys(), meta.coef_.round(3))))
print("Training RMSE (stack):", round(mean_squared_error(y, meta.predict(S_tr)) ** 0.5, 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta.predict(S_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v6_stack.csv', index=False)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.06GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

100%|██████████| 216/216 [00:39<00:00,  5.46it/s]


text: best alpha=10  CV RMSE=0.994
w2v: best alpha=1000  CV RMSE=0.642
wavlm: best alpha=1000  CV RMSE=0.564
whisper: best alpha=3000  CV RMSE=0.567
wavlm_large: best alpha=1000  CV RMSE=0.535
whisper_med: best alpha=1000  CV RMSE=0.547
Stack CV (RMSE, Pearson): (0.515, np.float64(0.909))
Block weights: {'text': np.float64(0.146), 'w2v': np.float64(0.024), 'wavlm': np.float64(0.102), 'whisper': np.float64(-0.11), 'wavlm_large': np.float64(0.509), 'whisper_med': np.float64(0.445)}
Training RMSE (stack): 0.511


In [21]:
alphas = {'text': 10, 'w2v': 1000, 'wavlm': 1000, 'whisper': 3000, 'wavlm_large': 1000, 'whisper_med': 1000}
P_tr = np.column_stack([Ridge(alpha=alphas[k]).fit(blocks_tr[k], y).predict(blocks_tr[k]) for k in blocks_tr])
print("Training RMSE (in-sample):", round(mean_squared_error(y, np.clip(meta.predict(P_tr), 0, 5)) ** 0.5, 3))

Training RMSE (in-sample): 0.254


In [23]:
for root, _, files in os.walk('/kaggle/input'):
    if 'A_tr.npy' in files:
        feat_dir = root
print(feat_dir)
A_tr = np.load(f'{feat_dir}/A_tr.npy')   # same for others

/kaggle/input/datasets/harshareddynayani/shl-features


In [24]:
feat_dir = '/kaggle/input/datasets/harshareddynayani/shl-features'

A_tr, A_te = np.load(f'{feat_dir}/A_tr.npy'), np.load(f'{feat_dir}/A_te.npy')   # wav2vec2
W_tr, W_te = np.load(f'{feat_dir}/W_tr.npy'), np.load(f'{feat_dir}/W_te.npy')   # WavLM-base
H_tr, H_te = np.load(f'{feat_dir}/H_tr.npy'), np.load(f'{feat_dir}/H_te.npy')   # Whisper-small
L_tr, L_te = np.load(f'{feat_dir}/L_tr.npy'), np.load(f'{feat_dir}/L_te.npy')   # WavLM-large
M_tr, M_te = np.load(f'{feat_dir}/M_tr.npy'), np.load(f'{feat_dir}/M_te.npy')   # Whisper-medium

transcripts = pd.read_csv(f'{feat_dir}/transcripts.csv').fillna('')

In [25]:
import gc
for v in ['wenc2', 'wl', 'wavlm', 'w2v', 'wenc']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

wfe3 = WhisperFeatureExtractor.from_pretrained('openai/whisper-large-v3')
wenc3 = WhisperModel.from_pretrained('openai/whisper-large-v3', torch_dtype=torch.float16).encoder.cuda().eval()

def whisper_large_emb(path):
    a, _ = librosa.load(path, sr=16000)
    chunks = [a[i:i + 480000] for i in range(0, len(a), 480000)]
    chunks = [c for c in chunks if len(c) > 16000] or [a]
    embs = []
    for c in chunks:
        n = max(int(len(c) / 16000 * 50), 2)
        x = wfe3(c, sampling_rate=16000, return_tensors='pt').input_features.cuda().half()
        with torch.no_grad():
            hs = wenc3(x, output_hidden_states=True).hidden_states
        f = []
        for l in [16, 24, 32]:
            h = hs[l][0, :n].float()
            f += [h.mean(0), h.std(0)]
        embs.append(torch.cat(f).cpu().numpy())
    return np.mean(embs, axis=0)

G_tr = np.array([whisper_large_emb(wav['train'][f]) for f in tqdm(tr.filename)])
G_te = np.array([whisper_large_emb(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/G_tr.npy', G_tr); np.save('/kaggle/working/G_te.npy', G_te)

sg = StandardScaler().fit(G_tr)
blocks_tr['whisper_large'] = sg.transform(G_tr)
blocks_te['whisper_large'] = sg.transform(G_te)

oof, tep = {}, {}
for k in blocks_tr:
    scores = [(cv(Ridge(alpha=a), blocks_tr[k])[0], a) for a in [10, 100, 1000, 3000, 10000]]
    rmse_k, a_k = min(scores)
    oof[k] = cross_val_predict(Ridge(alpha=a_k), blocks_tr[k], y, cv=kf)
    tep[k] = Ridge(alpha=a_k).fit(blocks_tr[k], y).predict(blocks_te[k])
    print(f"{k}: best alpha={a_k}  CV RMSE={rmse_k:.3f}")

S_tr = np.column_stack(list(oof.values()))
S_te = np.column_stack(list(tep.values()))
print("Stack CV (RMSE, Pearson):", cv(Ridge(alpha=1), S_tr))
meta = Ridge(alpha=1).fit(S_tr, y)
print("Block weights:", dict(zip(oof.keys(), meta.coef_.round(3))))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta.predict(S_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v7_stack.csv', index=False)

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

100%|██████████| 216/216 [01:09<00:00,  3.11it/s]


text: best alpha=10  CV RMSE=0.994
w2v: best alpha=1000  CV RMSE=0.642
wavlm: best alpha=1000  CV RMSE=0.564
whisper: best alpha=3000  CV RMSE=0.567
wavlm_large: best alpha=1000  CV RMSE=0.535
whisper_med: best alpha=1000  CV RMSE=0.547
whisper_large: best alpha=3000  CV RMSE=0.570
Stack CV (RMSE, Pearson): (0.515, np.float64(0.909))
Block weights: {'text': np.float64(0.148), 'w2v': np.float64(0.028), 'wavlm': np.float64(0.107), 'whisper': np.float64(-0.077), 'wavlm_large': np.float64(0.509), 'whisper_med': np.float64(0.463), 'whisper_large': np.float64(-0.062)}


In [2]:
!ls /kaggle/input

competitions  datasets


In [4]:
print([n for n in ['oof', 'tep', 'blocks_tr', 'y', 'kf', 'cv'] if n not in globals()])

['oof', 'tep', 'blocks_tr', 'y', 'kf', 'cv']


In [5]:
import os, re, numpy as np, pandas as pd, torch
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sentence_transformers import SentenceTransformer

base = next(r for r, _, f in os.walk('/kaggle/input') if 'train.csv' in f)
feat_dir = '/kaggle/input/datasets/harshareddynayani/shl-features'

train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test.csv')
t = pd.read_csv(f'{feat_dir}/transcripts.csv').fillna('')
tr = train.merge(t[t.split == 'train'][['filename', 'text']], on='filename', how='left')
te = test.merge(t[t.split == 'test'][['filename', 'text']], on='filename', how='left')
y = tr.label.values

# text block
emb_model = SentenceTransformer('all-mpnet-base-v2', device='cuda')
E_tr = emb_model.encode(tr.text.tolist(), batch_size=32); E_te = emb_model.encode(te.text.tolist(), batch_size=32)

def feats(s):
    w = s.split(); sents = [x for x in re.split(r'[.!?]+', s) if x.strip()]
    fill = len(re.findall(r'\b(um+|uh+|like|you know)\b', s.lower()))
    return [len(w), len(sents), len(w) / max(len(sents), 1), len(set(w)) / max(len(w), 1), fill,
            np.mean([len(x) for x in w]) if w else 0]
F_tr = np.array([feats(s) for s in tr.text]); F_te = np.array([feats(s) for s in te.text])
sc = StandardScaler().fit(F_tr)
blocks_tr = {'text': np.hstack([E_tr, sc.transform(F_tr)])}
blocks_te = {'text': np.hstack([E_te, sc.transform(F_te)])}

# audio blocks (saved embeddings)
for k, p in {'w2v': 'A', 'wavlm': 'W', 'whisper': 'H', 'wavlm_large': 'L', 'whisper_med': 'M'}.items():
    a_tr, a_te = np.load(f'{feat_dir}/{p}_tr.npy'), np.load(f'{feat_dir}/{p}_te.npy')
    s = StandardScaler().fit(a_tr)
    blocks_tr[k], blocks_te[k] = s.transform(a_tr), s.transform(a_te)

kf = KFold(5, shuffle=True, random_state=42)
def cv(model, X):
    oof_ = cross_val_predict(model, X, y, cv=kf)
    return round(mean_squared_error(y, oof_) ** 0.5, 3), round(pearsonr(y, oof_)[0], 3)

alphas, oof, tep = {}, {}, {}
for k in blocks_tr:
    rmse_k, a_k = min((cv(Ridge(alpha=a), blocks_tr[k])[0], a) for a in [10, 100, 1000, 3000, 10000])
    alphas[k] = a_k
    oof[k] = cross_val_predict(Ridge(alpha=a_k), blocks_tr[k], y, cv=kf)
    tep[k] = Ridge(alpha=a_k).fit(blocks_tr[k], y).predict(blocks_te[k])
    print(f"{k}: alpha={a_k}  CV RMSE={rmse_k}")

S_tr, S_te = np.column_stack(list(oof.values())), np.column_stack(list(tep.values()))
print("Stack CV (RMSE, Pearson):", cv(Ridge(alpha=1), S_tr))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

text: alpha=10  CV RMSE=0.994
w2v: alpha=1000  CV RMSE=0.642
wavlm: alpha=1000  CV RMSE=0.564
whisper: alpha=3000  CV RMSE=0.567
wavlm_large: alpha=1000  CV RMSE=0.535
whisper_med: alpha=1000  CV RMSE=0.547
Stack CV (RMSE, Pearson): (0.515, np.float64(0.909))


In [ ]:
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LinearRegression

oof2, tep2 = dict(oof), dict(tep)
for k in blocks_tr:
    Xtr, Xte = blocks_tr[k], blocks_te[k]
    g = 1.0 / Xtr.shape[1]
    cands = {f'svr_C{C}': (lambda C=C: SVR(C=C, gamma=g)) for C in [1, 3, 10]}
    cands |= {f'kr_a{a}': (lambda a=a: KernelRidge(alpha=a, kernel='rbf', gamma=g)) for a in [0.1, 1]}
    scores = {n: cv(m(), Xtr)[0] for n, m in cands.items()}
    best = min(scores, key=scores.get)
    print(k, best, scores[best])
    oof2[f'{k}_{best}'] = cross_val_predict(cands[best](), Xtr, y, cv=kf)
    tep2[f'{k}_{best}'] = cands[best]().fit(Xtr, y).predict(Xte)

S2_tr, S2_te = np.column_stack(list(oof2.values())), np.column_stack(list(tep2.values()))
meta2 = LinearRegression(positive=True).fit(S2_tr, y)     # non-negative weights, more stable
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S2_tr))
print(dict(zip(oof2.keys(), meta2.coef_.round(3))))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta2.predict(S2_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v8.csv', index=False)

In [6]:
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LinearRegression

oof2, tep2 = dict(oof), dict(tep)
for k in blocks_tr:
    Xtr, Xte = blocks_tr[k], blocks_te[k]
    g = 1.0 / Xtr.shape[1]
    cands = {f'svr_C{C}': (lambda C=C: SVR(C=C, gamma=g)) for C in [1, 3, 10]}
    cands |= {f'kr_a{a}': (lambda a=a: KernelRidge(alpha=a, kernel='rbf', gamma=g)) for a in [0.1, 1]}
    scores = {n: cv(m(), Xtr)[0] for n, m in cands.items()}
    best = min(scores, key=scores.get)
    print(k, best, scores[best])
    oof2[f'{k}_{best}'] = cross_val_predict(cands[best](), Xtr, y, cv=kf)
    tep2[f'{k}_{best}'] = cands[best]().fit(Xtr, y).predict(Xte)

S2_tr, S2_te = np.column_stack(list(oof2.values())), np.column_stack(list(tep2.values()))
meta2 = LinearRegression(positive=True).fit(S2_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S2_tr))
print(dict(zip(oof2.keys(), meta2.coef_.round(3))))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta2.predict(S2_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v8.csv', index=False)

text kr_a0.1 1.05
w2v svr_C10 0.628
wavlm svr_C10 0.551
whisper svr_C10 0.592
wavlm_large svr_C10 0.543
whisper_med svr_C10 0.569
Stack CV (RMSE, Pearson): (0.508, np.float64(0.912))
{'text': np.float64(0.126), 'w2v': np.float64(0.0), 'wavlm': np.float64(0.0), 'whisper': np.float64(0.0), 'wavlm_large': np.float64(0.247), 'whisper_med': np.float64(0.244), 'text_kr_a0.1': np.float64(0.0), 'w2v_svr_C10': np.float64(0.0), 'wavlm_svr_C10': np.float64(0.165), 'whisper_svr_C10': np.float64(0.0), 'wavlm_large_svr_C10': np.float64(0.192), 'whisper_med_svr_C10': np.float64(0.173)}


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, GPT2LMHeadModel, GPT2TokenizerFast

ctok = AutoTokenizer.from_pretrained('textattack/roberta-base-CoLA')
cola = AutoModelForSequenceClassification.from_pretrained('textattack/roberta-base-CoLA').cuda().eval().half()
gtok = GPT2TokenizerFast.from_pretrained('gpt2')
gpt = GPT2LMHeadModel.from_pretrained('gpt2').cuda().eval()

def gram_feats(text):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.split()) >= 3] or [text or 'empty']
    ps = []
    for i in range(0, len(sents), 32):
        enc = ctok(sents[i:i + 32], return_tensors='pt', padding=True, truncation=True, max_length=64).to('cuda')
        with torch.no_grad():
            ps += torch.softmax(cola(**enc).logits.float(), -1)[:, 1].cpu().tolist()   # P(acceptable)
    ps = np.array(ps)
    ids = gtok(text or 'empty', return_tensors='pt', truncation=True, max_length=512).input_ids.cuda()
    with torch.no_grad():
        lp = float(gpt(ids, labels=ids).loss) if ids.shape[1] > 1 else 5.0            # GPT-2 log-perplexity
    return [ps.mean(), ps.min(), np.median(ps), (ps > 0.5).mean(), lp, len(sents)]

G_tr = np.array([gram_feats(s) for s in tr.text]); G_te = np.array([gram_feats(s) for s in te.text])
np.save('/kaggle/working/Gram_tr.npy', G_tr); np.save('/kaggle/working/Gram_te.npy', G_te)

sg = StandardScaler().fit(np.hstack([G_tr, F_tr]))
blocks_tr['gram'] = sg.transform(np.hstack([G_tr, F_tr]))
blocks_te['gram'] = sg.transform(np.hstack([G_te, F_te]))

# best head for the new block: Ridge or SVR
cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [1, 10, 100]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [1, 3, 10]}
scores = {n: cv(m(), blocks_tr['gram'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('gram', best, scores[best])
oof3, tep3 = dict(oof2), dict(tep2)
oof3['gram'] = cross_val_predict(cands[best](), blocks_tr['gram'], y, cv=kf)
tep3['gram'] = cands[best]().fit(blocks_tr['gram'], y).predict(blocks_te['gram'])

S3_tr, S3_te = np.column_stack(list(oof3.values())), np.column_stack(list(tep3.values()))
meta3 = LinearRegression(positive=True).fit(S3_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S3_tr))
print("gram weight:", round(meta3.coef_[-1], 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta3.predict(S3_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v9.csv', index=False)

config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


gram ridge_a10 0.999
Stack CV (RMSE, Pearson): (0.502, np.float64(0.914))
gram weight: 0.171


In [8]:
cola2_tok = AutoTokenizer.from_pretrained('textattack/bert-base-uncased-CoLA')
cola2 = AutoModelForSequenceClassification.from_pretrained('textattack/bert-base-uncased-CoLA').cuda().eval().half()
gtok_m = GPT2TokenizerFast.from_pretrained('gpt2-medium')
gpt_m = GPT2LMHeadModel.from_pretrained('gpt2-medium').cuda().eval().half()

def cola_stats(tk, model, text):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.split()) >= 3] or [text or 'empty']
    ps = []
    for i in range(0, len(sents), 32):
        enc = tk(sents[i:i + 32], return_tensors='pt', padding=True, truncation=True, max_length=64).to('cuda')
        with torch.no_grad():
            ps += torch.softmax(model(**enc).logits.float(), -1)[:, 1].cpu().tolist()
    ps = np.array(ps)
    return [ps.mean(), ps.min(), np.median(ps), (ps > 0.5).mean()]

def tok_stats(model, tk, text):
    ids = tk(text or 'empty', return_tensors='pt', truncation=True, max_length=512).input_ids.cuda()
    if ids.shape[1] < 3:
        return [5.0] * 5
    with torch.no_grad():
        logits = model(ids).logits.float()
    loss = torch.nn.functional.cross_entropy(logits[0, :-1], ids[0, 1:], reduction='none').cpu().numpy()
    return [loss.mean(), loss.std(), np.percentile(loss, 90), (loss > 8).mean(), np.median(loss)]

def gram2(text):
    return cola_stats(cola2_tok, cola2, text) + tok_stats(gpt, gtok, text) + tok_stats(gpt_m, gtok_m, text)

G2_tr = np.array([gram2(s) for s in tqdm(tr.text)]); G2_te = np.array([gram2(s) for s in tqdm(te.text)])
np.save('/kaggle/working/Gram2_tr.npy', G2_tr); np.save('/kaggle/working/Gram2_te.npy', G2_te)

Xg_tr, Xg_te = np.hstack([G2_tr, G_tr, F_tr]), np.hstack([G2_te, G_te, F_te])
sg2 = StandardScaler().fit(Xg_tr)
blocks_tr['gram2'], blocks_te['gram2'] = sg2.transform(Xg_tr), sg2.transform(Xg_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [1, 10, 100]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [1, 3, 10]}
scores = {n: cv(m(), blocks_tr['gram2'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('gram2', best, scores[best])

oof4, tep4 = dict(oof3), dict(tep3)
oof4['gram2'] = cross_val_predict(cands[best](), blocks_tr['gram2'], y, cv=kf)
tep4['gram2'] = cands[best]().fit(blocks_tr['gram2'], y).predict(blocks_te['gram2'])

S4_tr, S4_te = np.column_stack(list(oof4.values())), np.column_stack(list(tep4.values()))
meta4 = LinearRegression(positive=True).fit(S4_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S4_tr))
print("gram / gram2 weights:", round(meta4.coef_[-2], 3), round(meta4.coef_[-1], 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta4.predict(S4_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v10.csv', index=False)

config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

NameError: name 'tqdm' is not defined

In [9]:
from tqdm import tqdm

G2_tr = np.array([gram2(s) for s in tqdm(tr.text)]); G2_te = np.array([gram2(s) for s in tqdm(te.text)])
np.save('/kaggle/working/Gram2_tr.npy', G2_tr); np.save('/kaggle/working/Gram2_te.npy', G2_te)

Xg_tr, Xg_te = np.hstack([G2_tr, G_tr, F_tr]), np.hstack([G2_te, G_te, F_te])
sg2 = StandardScaler().fit(Xg_tr)
blocks_tr['gram2'], blocks_te['gram2'] = sg2.transform(Xg_tr), sg2.transform(Xg_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [1, 10, 100]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [1, 3, 10]}
scores = {n: cv(m(), blocks_tr['gram2'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('gram2', best, scores[best])

oof4, tep4 = dict(oof3), dict(tep3)
oof4['gram2'] = cross_val_predict(cands[best](), blocks_tr['gram2'], y, cv=kf)
tep4['gram2'] = cands[best]().fit(blocks_tr['gram2'], y).predict(blocks_te['gram2'])

S4_tr, S4_te = np.column_stack(list(oof4.values())), np.column_stack(list(tep4.values()))
meta4 = LinearRegression(positive=True).fit(S4_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S4_tr))
print("gram / gram2 weights:", round(meta4.coef_[-2], 3), round(meta4.coef_[-1], 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta4.predict(S4_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v10.csv', index=False)

100%|██████████| 216/216 [00:12<00:00, 16.95it/s]


gram2 ridge_a10 0.968
Stack CV (RMSE, Pearson): (0.501, np.float64(0.915))
gram / gram2 weights: 0.057 0.124


In [10]:
import librosa
from tqdm import tqdm

# audio paths per split (train and test filenames overlap, so key by split)
wav = {'train': {}, 'test': {}}
for root, _, files in os.walk(base):
    sp = 'train' if '/train' in root else 'test' if '/test' in root else None
    if sp:
        for f in files:
            if f.endswith('.wav'):
                wav[sp][f] = os.path.join(root, f)

def fluency(path, text):
    a, sr = librosa.load(path, sr=16000)
    dur = len(a) / sr
    iv = librosa.effects.split(a, top_db=30)                       # non-silent intervals
    speech = max(sum(e - s for s, e in iv) / sr, 1e-3)
    gaps = [(iv[i + 1][0] - iv[i][1]) / sr for i in range(len(iv) - 1)]
    long = [g for g in gaps if g > 0.3]                            # pauses > 0.3 sec
    nw = len(text.split())
    rms = librosa.feature.rms(y=a)[0]
    return [dur, speech / dur, nw / dur, nw / speech, len(long) / dur * 60,
            np.mean(long) if long else 0, max(gaps) if gaps else 0,
            np.std(gaps) if gaps else 0, rms.mean(), rms.std()]

FL_tr = np.array([fluency(wav['train'][f], s) for f, s in tqdm(zip(tr.filename, tr.text), total=len(tr))])
FL_te = np.array([fluency(wav['test'][f], s) for f, s in tqdm(zip(te.filename, te.text), total=len(te))])
np.save('/kaggle/working/Flu_tr.npy', FL_tr); np.save('/kaggle/working/Flu_te.npy', FL_te)

Xf_tr, Xf_te = np.hstack([FL_tr, G2_tr, G_tr, F_tr]), np.hstack([FL_te, G2_te, G_te, F_te])
sf = StandardScaler().fit(Xf_tr)
blocks_tr['flu'], blocks_te['flu'] = sf.transform(Xf_tr), sf.transform(Xf_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [1, 10, 100]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [1, 3, 10]}
scores = {n: cv(m(), blocks_tr['flu'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('flu', best, scores[best])

oof5, tep5 = dict(oof4), dict(tep4)
oof5['flu'] = cross_val_predict(cands[best](), blocks_tr['flu'], y, cv=kf)
tep5['flu'] = cands[best]().fit(blocks_tr['flu'], y).predict(blocks_te['flu'])

S5_tr, S5_te = np.column_stack(list(oof5.values())), np.column_stack(list(tep5.values()))
meta5 = LinearRegression(positive=True).fit(S5_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S5_tr))
print("flu weight:", round(meta5.coef_[-1], 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta5.predict(S5_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v11.csv', index=False)

100%|██████████| 216/216 [00:48<00:00,  4.49it/s]


flu svr_C1 0.743
Stack CV (RMSE, Pearson): (0.495, np.float64(0.917))
flu weight: 0.145


In [11]:
def fluency2(path):
    a, sr = librosa.load(path, sr=16000)
    dur = len(a) / sr
    out = [dur]
    for db in [20, 25, 35, 40]:                                  # multiple silence thresholds
        iv = librosa.effects.split(a, top_db=db)
        speech = max(sum(e - s for s, e in iv) / sr, 1e-3)
        gaps = [(iv[i + 1][0] - iv[i][1]) / sr for i in range(len(iv) - 1)]
        out += [speech / dur, len(iv) / dur] + [sum(g > th for g in gaps) / dur * 60 for th in (0.2, 0.5, 1.0)] \
               + [np.mean(gaps) if gaps else 0]
    mf = librosa.feature.mfcc(y=a, sr=sr, n_mfcc=20)             # voice / pronunciation shape
    f0 = librosa.yin(a, fmin=60, fmax=400, sr=sr)                # pitch (intonation)
    sc = librosa.feature.spectral_centroid(y=a, sr=sr)[0]; zc = librosa.feature.zero_crossing_rate(a)[0]
    out += list(mf.mean(1)) + list(mf.std(1)) + [f0.mean(), f0.std(), np.percentile(f0, 90) - np.percentile(f0, 10)]
    out += [sc.mean(), sc.std(), zc.mean(), zc.std()]
    return out

FL2_tr = np.array([fluency2(wav['train'][f]) for f in tqdm(tr.filename)])
FL2_te = np.array([fluency2(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/Flu2_tr.npy', FL2_tr); np.save('/kaggle/working/Flu2_te.npy', FL2_te)

Xf2_tr, Xf2_te = np.hstack([FL2_tr, FL_tr, G2_tr, G_tr, F_tr]), np.hstack([FL2_te, FL_te, G2_te, G_te, F_te])
sf2 = StandardScaler().fit(Xf2_tr)
blocks_tr['flu2'], blocks_te['flu2'] = sf2.transform(Xf2_tr), sf2.transform(Xf2_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [10, 100, 300]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [0.3, 1, 3]}
scores = {n: cv(m(), blocks_tr['flu2'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('flu2', best, scores[best])

oof6, tep6 = dict(oof5), dict(tep5)
oof6['flu2'] = cross_val_predict(cands[best](), blocks_tr['flu2'], y, cv=kf)
tep6['flu2'] = cands[best]().fit(blocks_tr['flu2'], y).predict(blocks_te['flu2'])

S6_tr, S6_te = np.column_stack(list(oof6.values())), np.column_stack(list(tep6.values()))
meta6 = LinearRegression(positive=True).fit(S6_tr, y)
print("Stack CV (RMSE, Pearson):", cv(LinearRegression(positive=True), S6_tr))
print("flu / flu2 weights:", round(meta6.coef_[-2], 3), round(meta6.coef_[-1], 3))
pd.DataFrame({'filename': te.filename, 'label': np.clip(meta6.predict(S6_te), 0, 5)}) \
  .to_csv('/kaggle/working/submission_v12.csv', index=False)

100%|██████████| 216/216 [00:59<00:00,  3.62it/s]


flu2 svr_C3 0.645
Stack CV (RMSE, Pearson): (0.493, np.float64(0.917))
flu / flu2 weights: 0.048 0.152


In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor

Xc_tr = np.hstack([S6_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr])
Xc_te = np.hstack([S6_te, FL2_te, FL_te, G2_te, G_te, F_te])
gb = lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, max_depth=3,
                                           min_samples_leaf=15, l2_regularization=1.0, random_state=0)
print("GB meta CV (RMSE, Pearson):", cv(gb(), Xc_tr))

oof_lin = cross_val_predict(LinearRegression(positive=True), S6_tr, y, cv=kf)
oof_gb = cross_val_predict(gb(), Xc_tr, y, cv=kf)
best_w, best_r = 0, 9
for w in [0, 0.2, 0.3, 0.5, 0.7, 1]:
    b = np.clip(w * oof_gb + (1 - w) * oof_lin, 0, 5)
    r = mean_squared_error(y, b) ** 0.5
    print(f"w_gb={w}  CV RMSE={r:.3f}  Pearson={pearsonr(y, b)[0]:.3f}")
    if r < best_r: best_w, best_r = w, r

p_lin = meta6.predict(S6_te)
p_gb = gb().fit(Xc_tr, y).predict(Xc_te)
pred = np.clip(best_w * p_gb + (1 - best_w) * p_lin, 0, 5)
pd.DataFrame({'filename': te.filename, 'label': pred}).to_csv('/kaggle/working/submission_v13.csv', index=False)
print("best w_gb:", best_w, "CV RMSE:", round(best_r, 3))

GB meta CV (RMSE, Pearson): (0.505, np.float64(0.913))
w_gb=0  CV RMSE=0.489  Pearson=0.919
w_gb=0.2  CV RMSE=0.485  Pearson=0.920
w_gb=0.3  CV RMSE=0.484  Pearson=0.921
w_gb=0.5  CV RMSE=0.485  Pearson=0.920
w_gb=0.7  CV RMSE=0.490  Pearson=0.918
w_gb=1  CV RMSE=0.505  Pearson=0.913
best w_gb: 0.3 CV RMSE: 0.484


In [14]:
# restack + GB blend (same as v13)
S7_tr = np.column_stack(list(oof7.values()))
S7_te = np.column_stack(list(tep7.values()))
meta7 = LinearRegression(positive=True).fit(S7_tr, y)

Xd_tr = np.hstack([S7_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr])
Xd_te = np.hstack([S7_te, FL2_te, FL_te, G2_te, G_te, F_te])

oof_lin = cross_val_predict(LinearRegression(positive=True), S7_tr, y, cv=kf)
oof_gb = cross_val_predict(gb(), Xd_tr, y, cv=kf)

best_w, best_r = 0, 9
for w in [0, 0.2, 0.3, 0.5]:
    r = mean_squared_error(y, np.clip(w * oof_gb + (1 - w) * oof_lin, 0, 5)) ** 0.5
    print(f"w_gb={w}  CV RMSE={r:.3f}")
    if r < best_r:
        best_w, best_r = w, r

p_gb = gb().fit(Xd_tr, y).predict(Xd_te)
pred = np.clip(best_w * p_gb + (1 - best_w) * meta7.predict(S7_te), 0, 5)
pd.DataFrame({'filename': te.filename, 'label': pred}).to_csv('/kaggle/working/submission_v14.csv', index=False)
print("v14 blend CV RMSE:", round(best_r, 3), "| wl_layers weight:", round(meta7.coef_[-1], 3))

w_gb=0  CV RMSE=0.490
w_gb=0.2  CV RMSE=0.486
w_gb=0.3  CV RMSE=0.485
w_gb=0.5  CV RMSE=0.485
v14 blend CV RMSE: 0.485 | wl_layers weight: 0.123


In [15]:
!pip install -q faster-whisper
import gc
from faster_whisper import WhisperModel
for v in ['wl_m']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

wsm = WhisperModel("small", device="cuda", compute_type="float16")

def asr_feats(path):
    a, _ = librosa.load(path, sr=16000)
    dur = len(a) / 16000
    segs, _ = wsm.transcribe(a, language="en", beam_size=1, word_timestamps=True, condition_on_previous_text=False)
    segs = list(segs)
    words = [w for s in segs for w in (s.words or [])]
    if len(words) < 3:
        return [dur, len(words)] + [0.0] * 20
    p = np.array([w.probability for w in words])
    st = np.array([w.start for w in words]); en = np.array([w.end for w in words])
    gaps = st[1:] - en[:-1]
    txt = [w.word.strip().lower() for w in words]
    rep = np.mean([txt[i] == txt[i + 1] for i in range(len(txt) - 1)])
    lp = np.array([s.avg_logprob for s in segs]); cr = np.array([s.compression_ratio for s in segs])
    ns = np.array([s.no_speech_prob for s in segs])
    return [dur, len(words), len(words) / max(en[-1] - st[0], 1e-3),
            p.mean(), p.min(), p.std(), np.percentile(p, 10), (p < 0.5).mean(), (p < 0.8).mean(),
            lp.mean(), lp.min(), cr.mean(), cr.max(), ns.mean(),
            gaps.mean(), gaps.std(), (gaps > 0.5).mean(), (gaps > 1.0).sum() / dur * 60,
            rep, st[0], dur - en[-1], np.clip(gaps, 0, None).max()]

AS_tr = np.array([asr_feats(wav['train'][f]) for f in tqdm(tr.filename)])
AS_te = np.array([asr_feats(wav['test'][f]) for f in tqdm(te.filename)])
np.save('/kaggle/working/Asr_tr.npy', AS_tr); np.save('/kaggle/working/Asr_te.npy', AS_te)

Xa_tr, Xa_te = np.hstack([AS_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr]), np.hstack([AS_te, FL2_te, FL_te, G2_te, G_te, F_te])
sa_ = StandardScaler().fit(Xa_tr)
blocks_tr['asr'], blocks_te['asr'] = sa_.transform(Xa_tr), sa_.transform(Xa_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [10, 100, 300]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [0.3, 1, 3]}
scores = {n: cv(m(), blocks_tr['asr'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('asr', best, scores[best], "(flu2 block was 0.645)")

oof8, tep8 = dict(oof6), dict(tep6)
oof8['asr'] = cross_val_predict(cands[best](), blocks_tr['asr'], y, cv=kf)
tep8['asr'] = cands[best]().fit(blocks_tr['asr'], y).predict(blocks_te['asr'])

S8_tr = np.column_stack(list(oof8.values()))
S8_te = np.column_stack(list(tep8.values()))
meta8 = LinearRegression(positive=True).fit(S8_tr, y)

Xe_tr = np.hstack([S8_tr, AS_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr])
Xe_te = np.hstack([S8_te, AS_te, FL2_te, FL_te, G2_te, G_te, F_te])
oof_lin = cross_val_predict(LinearRegression(positive=True), S8_tr, y, cv=kf)
oof_gb = cross_val_predict(gb(), Xe_tr, y, cv=kf)
best_w, best_r = 0, 9
for w in [0, 0.2, 0.3, 0.5]:
    r = mean_squared_error(y, np.clip(w * oof_gb + (1 - w) * oof_lin, 0, 5)) ** 0.5
    print(f"w_gb={w}  CV RMSE={r:.3f}")
    if r < best_r:
        best_w, best_r = w, r

p_gb = gb().fit(Xe_tr, y).predict(Xe_te)
pred = np.clip(best_w * p_gb + (1 - best_w) * meta8.predict(S8_te), 0, 5)
pd.DataFrame({'filename': te.filename, 'label': pred}).to_csv('/kaggle/working/submission_v15.csv', index=False)
print("v15 blend CV RMSE:", round(best_r, 3), "| asr weight:", round(meta8.coef_[-1], 3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 56.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 47.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.7 MB/s eta 0:00:00:00:0100:01


100%|██████████| 216/216 [02:19<00:00,  1.54it/s]


asr ridge_a100 0.626 (flu2 block was 0.645)
w_gb=0  CV RMSE=0.489
w_gb=0.2  CV RMSE=0.486
w_gb=0.3  CV RMSE=0.486
w_gb=0.5  CV RMSE=0.488
v15 blend CV RMSE: 0.486 | asr weight: 0.064


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
for v in ['wsm', 'wl_m', 'gpt', 'gpt_m', 'cola', 'cola2', 'emb_model']:     # free GPU memory
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

name = 'Qwen/Qwen2.5-3B-Instruct'
qtok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()
digit_ids = [qtok.encode(str(d), add_special_tokens=False)[0] for d in range(1, 6)]

RUBRIC = ("Rate the grammar of this spoken-English transcript from 1 to 5. "
          "1 = limited control of simple structures, incomplete sentences; 2 = frequent basic errors; "
          "3 = decent structure but noticeable errors; 4 = strong control, only occasional minor errors; "
          "5 = high accuracy, complex grammar used well. Ignore punctuation and spelling. Answer with one digit only.")

def llm_rate(text):
    text = ' '.join((text or 'empty').split()[:350])
    msgs = [{"role": "system", "content": "You are an expert English language examiner."},
            {"role": "user", "content": f"{RUBRIC}\n\nTranscript:\n{text}\n\nGrammar score (1-5):"}]
    prompt = qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = qtok(prompt, return_tensors='pt').input_ids.to('cuda')
    with torch.no_grad():
        logits = llm(ids).logits[0, -1].float()
    p = torch.softmax(logits[digit_ids], -1).cpu().numpy()
    return [float((p * np.arange(1, 6)).sum()), *p.tolist()]

LM_tr = np.array([llm_rate(s) for s in tqdm(tr.text)])
LM_te = np.array([llm_rate(s) for s in tqdm(te.text)])
np.save('/kaggle/working/Llm_tr.npy', LM_tr); np.save('/kaggle/working/Llm_te.npy', LM_te)
print("Pearson of LLM expected score vs label:", round(pearsonr(LM_tr[:, 0], y)[0], 3))

Xm_tr, Xm_te = np.hstack([LM_tr, G2_tr, G_tr, F_tr]), np.hstack([LM_te, G2_te, G_te, F_te])
sm_ = StandardScaler().fit(Xm_tr)
blocks_tr['llm'], blocks_te['llm'] = sm_.transform(Xm_tr), sm_.transform(Xm_te)

cands = {f'ridge_a{a}': (lambda a=a: Ridge(alpha=a)) for a in [10, 100, 300]}
cands |= {f'svr_C{C}': (lambda C=C: SVR(C=C)) for C in [0.3, 1, 3]}
scores = {n: cv(m(), blocks_tr['llm'])[0] for n, m in cands.items()}
best = min(scores, key=scores.get); print('llm', best, scores[best], "(gram2 block was 0.968)")

oof9, tep9 = dict(oof6), dict(tep6)          # v13 base + new llm block
oof9['llm'] = cross_val_predict(cands[best](), blocks_tr['llm'], y, cv=kf)
tep9['llm'] = cands[best]().fit(blocks_tr['llm'], y).predict(blocks_te['llm'])

S9_tr = np.column_stack(list(oof9.values()))
S9_te = np.column_stack(list(tep9.values()))
meta9 = LinearRegression(positive=True).fit(S9_tr, y)

Xl_tr = np.hstack([S9_tr, LM_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr])
Xl_te = np.hstack([S9_te, LM_te, FL2_te, FL_te, G2_te, G_te, F_te])
oof_lin = cross_val_predict(LinearRegression(positive=True), S9_tr, y, cv=kf)
oof_gb = cross_val_predict(gb(), Xl_tr, y, cv=kf)
best_w, best_r = 0, 9
for w in [0, 0.2, 0.3, 0.5]:
    r = mean_squared_error(y, np.clip(w * oof_gb + (1 - w) * oof_lin, 0, 5)) ** 0.5
    print(f"w_gb={w}  CV RMSE={r:.3f}")
    if r < best_r:
        best_w, best_r = w, r

p_gb = gb().fit(Xl_tr, y).predict(Xl_te)
pred = np.clip(best_w * p_gb + (1 - best_w) * meta9.predict(S9_te), 0, 5)
pd.DataFrame({'filename': te.filename, 'label': pred}).to_csv('/kaggle/working/submission_v16.csv', index=False)
print("v16 blend CV RMSE:", round(best_r, 3), "| llm weight:", round(meta9.coef_[-1], 3))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

100%|██████████| 216/216 [00:24<00:00,  8.67it/s]


Pearson of LLM expected score vs label: 0.427
llm ridge_a100 0.962 (gram2 block was 0.968)
w_gb=0  CV RMSE=0.489
w_gb=0.2  CV RMSE=0.485
w_gb=0.3  CV RMSE=0.485
w_gb=0.5  CV RMSE=0.486
v16 blend CV RMSE: 0.485 | llm weight: 0.005


In [20]:
import gc
for v in ['llm', 'qtok']:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

fe_l = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-large')
wl_m = AutoModel.from_pretrained('microsoft/wavlm-large').cuda().eval().half()

def frames(path):
    a, _ = librosa.load(path, sr=16000)
    x = fe_l(a, sampling_rate=16000, return_tensors='pt').input_values.cuda().half()
    with torch.no_grad():
        hs = wl_m(x, output_hidden_states=True).hidden_states
    h = torch.stack([hs[l][0] for l in (17, 19, 20, 21)]).float().mean(0)   # best layers, (T, 1024)
    T = h.shape[0] // 4 * 4
    return h[:T].reshape(-1, 4, 1024).mean(1).half().cpu()                  # pool 4 frames, ~12 fps

FR_tr = [frames(wav['train'][f]) for f in tqdm(tr.filename)]
FR_te = [frames(wav['test'][f]) for f in tqdm(te.filename)]
print(len(FR_tr), len(FR_te), FR_tr[0].shape)

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

100%|██████████| 216/216 [02:22<00:00,  1.51it/s]

769 216 torch.Size([756, 1024])


In [21]:
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

class Head(nn.Module):
    def __init__(s, d=1024, h=128):
        super().__init__()
        s.ln = nn.LayerNorm(d)
        s.proj = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, h), nn.GELU())
        s.gru = nn.GRU(h, h, batch_first=True, bidirectional=True)
        s.att = nn.Linear(2 * h, 1)
        s.out = nn.Sequential(nn.Dropout(0.3), nn.Linear(4 * h, 1))
    def forward(s, X, mask, L):
        z = s.proj(s.ln(X))
        z, _ = s.gru(pack_padded_sequence(z, L, batch_first=True, enforce_sorted=False))
        z, _ = pad_packed_sequence(z, batch_first=True, total_length=X.shape[1])
        a = s.att(z).squeeze(-1).masked_fill(~mask, -1e4).softmax(1)
        att = (a.unsqueeze(-1) * z).sum(1)
        mean = (z * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        return s.out(torch.cat([att, mean], 1)).squeeze(-1)

def make_batch(seqs, idx):
    xs = [seqs[i] for i in idx]
    L = torch.tensor([len(x) for x in xs])
    X = pad_sequence(xs, batch_first=True).float().cuda()
    mask = (torch.arange(X.shape[1])[None, :] < L[:, None]).cuda()
    return X, mask, L

def fit_predict(tr_idx, va_idx, seed, epochs=20):
    torch.manual_seed(seed); np.random.seed(seed)
    m = Head().cuda()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=epochs * int(np.ceil(len(tr_idx) / 16)))
    mu, sd = y[tr_idx].mean(), y[tr_idx].std()
    yt = torch.tensor((y - mu) / sd, dtype=torch.float32)
    for ep in range(epochs):
        m.train(); perm = np.random.permutation(tr_idx)
        for i in range(0, len(perm), 16):
            b = perm[i:i + 16]
            X, mask, L = make_batch(FR_tr, b)
            loss = nn.functional.smooth_l1_loss(m(X, mask, L), yt[b].cuda())
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step()
    m.eval()
    def pred(seqs, idx):
        out = []
        with torch.no_grad():
            for i in range(0, len(idx), 32):
                X, mask, L = make_batch(seqs, idx[i:i + 32]); out.append(m(X, mask, L).cpu().numpy())
        return np.concatenate(out) * sd + mu
    return pred(FR_tr, va_idx), pred(FR_te, np.arange(len(FR_te)))

n = len(y); oof_nn = np.zeros(n); tep_nn = np.zeros(len(FR_te)); cnt = 0
seeds = [0, 1]
for f, (a, b) in enumerate(kf.split(np.arange(n))):      # same folds as every other block
    for s in seeds:
        pv, pt = fit_predict(a, b, s)
        oof_nn[b] += pv / len(seeds); tep_nn += pt; cnt += 1
    print(f"fold {f} RMSE", round(mean_squared_error(y[b], oof_nn[b]) ** 0.5, 3))
tep_nn /= cnt
print("NN head CV RMSE / Pearson:", round(mean_squared_error(y, oof_nn) ** 0.5, 3),
      round(pearsonr(y, oof_nn)[0], 3), "(wavlm_large Ridge block was 0.535)")
np.save('/kaggle/working/nn_oof.npy', oof_nn); np.save('/kaggle/working/nn_te.npy', tep_nn)

# restack on top of v13 base + blend with GB (same as v13)
oof10, tep10 = dict(oof6), dict(tep6)
oof10['nn'], tep10['nn'] = oof_nn, tep_nn
S10_tr = np.column_stack(list(oof10.values()))
S10_te = np.column_stack(list(tep10.values()))
meta10 = LinearRegression(positive=True).fit(S10_tr, y)
Xn_tr = np.hstack([S10_tr, FL2_tr, FL_tr, G2_tr, G_tr, F_tr])
Xn_te = np.hstack([S10_te, FL2_te, FL_te, G2_te, G_te, F_te])
oof_lin = cross_val_predict(LinearRegression(positive=True), S10_tr, y, cv=kf)
oof_gb = cross_val_predict(gb(), Xn_tr, y, cv=kf)
best_w, best_r = 0, 9
for w in [0, 0.2, 0.3, 0.5]:
    r = mean_squared_error(y, np.clip(w * oof_gb + (1 - w) * oof_lin, 0, 5)) ** 0.5
    print(f"w_gb={w}  CV RMSE={r:.3f}")
    if r < best_r:
        best_w, best_r = w, r
p_gb = gb().fit(Xn_tr, y).predict(Xn_te)
pred = np.clip(best_w * p_gb + (1 - best_w) * meta10.predict(S10_te), 0, 5)
pd.DataFrame({'filename': te.filename, 'label': pred}).to_csv('/kaggle/working/submission_v17.csv', index=False)
print("v17 blend CV RMSE:", round(best_r, 3), "| nn weight:", round(meta10.coef_[-1], 3))

fold 0 RMSE 0.599
fold 1 RMSE 0.545
fold 2 RMSE 0.61
fold 3 RMSE 0.525
fold 4 RMSE 0.6
NN head CV RMSE / Pearson: 0.577 0.893 (wavlm_large Ridge block was 0.535)
w_gb=0  CV RMSE=0.489
w_gb=0.2  CV RMSE=0.484
w_gb=0.3  CV RMSE=0.483
w_gb=0.5  CV RMSE=0.484
v17 blend CV RMSE: 0.483 | nn weight: 0.116


In [ ]:
!ls -la /kaggle/working